### Gradient Descent with Nd4j 

In [1]:
import deeplearning_ai.machine_learning_specialization.course1.week2.with_nd4j.*;

import org.nd4j.linalg.api.ndarray.INDArray;
import org.nd4j.linalg.factory.Nd4j;
import org.nd4j.linalg.ops.transforms.Transforms;

import java.util.function.Function;
import static java.lang.System.out;

<img src="./images/020_multiple_feature.png" alt="Multiple Feature Sample" width="1024"> <br>

In [3]:
// Our Model : 
INDArray weights = Nd4j.create(new double[]{0.1, 0.2, 0.3, 0.4});    //Initial weights
double bias = 0.5;                                                   //Initial bias
MultivariableFunction ourModel = OurModel.create(weights, bias);

// our dataset features and targets (prices) : 
double[][] features = {
    // {sizeOfHouse, bedroomCount, floorCount, ageOfHouse}
    {2104., 5., 1., 45.},    // <--- First house
    {1416., 3., 2., 40.},
    {1534., 3., 2., 30.},
    {852.,  2., 1., 36.}
};
INDArray featureMatrix = Nd4j.create(features);
INDArray targets = Nd4j.create(new double[]{460., 232., 315., 178.});       // <--- prices 

out.println("Our model's parameters: weights = " + weights + " and bias = " + bias);
out.println("Our dataset Feature Matrix = \n" + featureMatrix);
out.println("Our dataset targets = " + targets);

// Our model's parameters: weights = [    0.1000,    0.2000,    0.3000,    0.4000] and bias = 0.5
// Our dataset Feature Matrix = 
// [[ 2104.0000,    5.0000,    1.0000,   45.0000], 
//  [ 1416.0000,    3.0000,    2.0000,   40.0000], 
//  [ 1534.0000,    3.0000,    2.0000,   30.0000], 
//  [  852.0000,    2.0000,    1.0000,   36.0000]]
// Our dataset targets = [  460.0000,  232.0000,  315.0000,  178.0000]

Our model's parameters: weights = [    0.1000,    0.2000,    0.3000,    0.4000] and bias = 0.5
Our dataset Feature Matrix = 
[[ 2104.0000,    5.0000,    1.0000,   45.0000], 
 [ 1416.0000,    3.0000,    2.0000,   40.0000], 
 [ 1534.0000,    3.0000,    2.0000,   30.0000], 
 [  852.0000,    2.0000,    1.0000,   36.0000]]
Our dataset targets = [  460.0000,  232.0000,  315.0000,  178.0000]


In [4]:
// Our Cost Function :
double J(){
    INDArray outputVector = ourModel.applyAll(featureMatrix);

    double sum = Transforms.pow(outputVector.sub(targets), 2).sumNumber().doubleValue();

    int m = (int) targets.length();
    return sum / (2*m);
}


out.println("J(" + ourModel.getWeights() + ", " + ourModel.getBias() + ") = " + J());

J([    0.1000,    0.2000,    0.3000,    0.4000], 0.5) = 10740.947499999998


We calculated derivatives ([week_2 notebook 020](./020_Derivatives.ipynb)) : <br>

$$
\frac{\partial J(\vec{w},b)}{\partial w_j} = \frac{1}{m}\cdot \sum_{i = 1}^{m} (f(\vec{x}^{(i)}) - y^{(i)}) \cdot x_j^{(i)}
$$

$$
\frac{\partial J(\vec{w},b)}{\partial b} = \frac{1}{m}\cdot \sum_{i = 1}^{m} f(\vec{x}^{(i)}) - y^{(i)}
$$
<br>

In [5]:
// Has been applied Vectorization here.
// If j < 0, calculates dJ/db, otherwise dJ/dw_j 
Function<Integer,Double> derivative = j -> {
    int m = (int) targets.length();

    INDArray resultVector = ourModel.applyAll(featureMatrix).sub(targets);

    // For dJ/dw_j calculate : 
    if (j > -1){
        INDArray jThFeaturesVector = featureMatrix.getColumn(j);
        resultVector = resultVector.mul(jThFeaturesVector);
    }

    double result = resultVector.sumNumber().doubleValue();
    return result / m;
};

#### Gradient Descent Step : 

In [6]:
void applyGradientDescent(int loopCount, double learningRate) {
    out.println("Initial J(" + ourModel.getWeights() + ", " + ourModel.getBias() + ") = " + J());

    for (int loop = 0; loop < loopCount; loop++) {
        int n = (int) ourModel.getWeights().length();  

        INDArray newWeights = ourModel.getWeights().dup(); ;
        // New Parameter Values : 
        for (int j = 0; j < n; j++){
            double dJ_dwj = (double) derivative.apply(j);
            double newWeight_j = ourModel.getWeights().getDouble(j) - learningRate * dJ_dwj;
            newWeights.putScalar(j, newWeight_j);
        }
        double new_b = ourModel.getBias() - learningRate * (double)derivative.apply(-1);

        // Updating Model's parameters : 
        ourModel.updateWeights(newWeights);
        ourModel.updateBias(new_b);

        // Print new J :
        out.println("Loop " + (loop + 1) +  " : J(" + ourModel.getWeights() + ", " + ourModel.getBias() + ") = " + J());
    }
}

In [ ]:
int loopCount = 4;
double learningRate = 5e-7;

applyGradientDescent(loopCount, learningRate);

// Logs : 
// Initial  J([    0.1000,    0.2000,    0.3000,    0.4000], 0.5)                = 10740.947499999998
// Loop 1 : J([    0.2099,    0.2002,    0.3001,    0.4026], 0.50006595)         = 949.3692856951917
// Loop 2 : J([    0.1891,    0.2002,    0.3001,    0.4020], 0.5000507271902375) = 597.527796933896
// Loop 3 : J([    0.1930,    0.2002,    0.3001,    0.4020], 0.5000508913734323) = 584.8740638725292
// Loop 4 : J([    0.1923,    0.2002,    0.3001,    0.4019], 0.5000481389497902) = 584.407995158359

Initial J([    0.1000,    0.2000,    0.3000,    0.4000], 0.5) = 10740.947499999998
Loop 1 : J([    0.2099,    0.2002,    0.3001,    0.4026], 0.50006595) = 949.3692856951917
Loop 2 : J([    0.1891,    0.2002,    0.3001,    0.4020], 0.5000507271902375) = 597.527796933896
Loop 3 : J([    0.1930,    0.2002,    0.3001,    0.4020], 0.5000508913734323) = 584.8740638725292
Loop 4 : J([    0.1923,    0.2002,    0.3001,    0.4019], 0.5000481389497902) = 584.407995158359


We chosed $\ learningRate = 5\cdot 10^{-7}.\ $ If we apply ***scaling on dataset***, we can choose $\ learningRate = 0.1\ or\ 0.01\ $